# EDA — Planta de Flotación Minera
## Quality Prediction in a Mining Process

**Autor:** Rodolfo Gabriel Riveros Lobos  
**Fecha:** Junio 2026  
**Dataset:** Kaggle — Eduardo Magalhães  
**Stack:** Python · Pandas · Matplotlib · Seaborn  

---

## Contexto del Caso

Una planta de flotación minera requiere análisis exploratorio de sus 
datos operativos. El objetivo es entender el comportamiento de las 
variables de calidad del concentrado final (% hierro, % sílice) e 
identificar qué variables de proceso presentan mayor correlación 
con la calidad del producto.

Este análisis simula un escenario real de Data Analyst Junior 
en operaciones mineras argentinas.

---

## Pregunta de Negocio

¿Qué variables operativas tienen mayor impacto sobre la calidad 
del concentrado final, y existen períodos de inestabilidad 
que requieran atención?

## 1. Importación de Librerías  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("Librerías cargadas correctamente.")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Seaborn: {sns.__version__}")

Librerías cargadas correctamente.
Pandas: 3.0.3
NumPy: 2.4.6
Seaborn: 0.13.2


## 2. Carga del Dataset

In [2]:
import os

ruta = "../data/raw/MiningProcess_Flotation_Plant_Database.csv"

assert os.path.exists(ruta), \
    f"Archivo no encontrado en: {ruta}"

# Separador: coma | Decimal: coma (entre comillas en el CSV)
df = pd.read_csv(ruta, sep=",", decimal=",")

assert df.shape[0] > 0, \
    "El dataframe está vacío."
assert df.shape[1] > 1, \
    f"Solo se cargó 1 columna. Separador incorrecto."

print(f"✓ Dataset cargado correctamente")
print(f"✓ Filas: {df.shape[0]:,}")
print(f"✓ Columnas: {df.shape[1]}")
print(f"\nPrimeras columnas detectadas:")
print(df.columns.tolist())

✓ Dataset cargado correctamente
✓ Filas: 737,453
✓ Columnas: 24

Primeras columnas detectadas:
['date', '% Iron Feed', '% Silica Feed', 'Starch Flow', 'Amina Flow', 'Ore Pulp Flow', 'Ore Pulp pH', 'Ore Pulp Density', 'Flotation Column 01 Air Flow', 'Flotation Column 02 Air Flow', 'Flotation Column 03 Air Flow', 'Flotation Column 04 Air Flow', 'Flotation Column 05 Air Flow', 'Flotation Column 06 Air Flow', 'Flotation Column 07 Air Flow', 'Flotation Column 01 Level', 'Flotation Column 02 Level', 'Flotation Column 03 Level', 'Flotation Column 04 Level', 'Flotation Column 05 Level', 'Flotation Column 06 Level', 'Flotation Column 07 Level', '% Iron Concentrate', '% Silica Concentrate']


## 3. Primer Vistazo al Dataset

In [3]:
# Dimensiones
print(f"Shape: {df.shape}")
print(f"\n--- Tipos de datos ---")
print(df.dtypes)
print(f"\n--- Primeras 3 filas ---")
df.head(3)

Shape: (737453, 24)

--- Tipos de datos ---
date                                str
% Iron Feed                     float64
% Silica Feed                   float64
Starch Flow                     float64
Amina Flow                      float64
Ore Pulp Flow                   float64
Ore Pulp pH                     float64
Ore Pulp Density                float64
Flotation Column 01 Air Flow    float64
Flotation Column 02 Air Flow    float64
Flotation Column 03 Air Flow    float64
Flotation Column 04 Air Flow    float64
Flotation Column 05 Air Flow    float64
Flotation Column 06 Air Flow    float64
Flotation Column 07 Air Flow    float64
Flotation Column 01 Level       float64
Flotation Column 02 Level       float64
Flotation Column 03 Level       float64
Flotation Column 04 Level       float64
Flotation Column 05 Level       float64
Flotation Column 06 Level       float64
Flotation Column 07 Level       float64
% Iron Concentrate              float64
% Silica Concentrate            floa

,date,% Iron Feed,% Silica Feed,Starch Flow,Amina Flow,Ore Pulp Flow,Ore Pulp pH,Ore Pulp Density,Flotation Column 01 Air Flow,Flotation Column 02 Air Flow,...,Flotation Column 07 Air Flow,Flotation Column 01 Level,Flotation Column 02 Level,Flotation Column 03 Level,Flotation Column 04 Level,Flotation Column 05 Level,Flotation Column 06 Level,Flotation Column 07 Level,% Iron Concentrate,% Silica Concentrate
0,2017-03-10 01:00:00,55.2,16.98,3019.53,557.434,395.713,10.0664,1.74,249.214,253.235,...,250.884,457.396,432.962,424.954,443.558,502.255,446.370,523.344,66.91,1.31
1,2017-03-10 01:00:00,55.2,16.98,3024.41,563.965,397.383,10.0672,1.74,249.719,250.532,...,248.994,451.891,429.560,432.939,448.086,496.363,445.922,498.075,66.91,1.31
2,2017-03-10 01:00:00,55.2,16.98,3043.46,568.054,399.668,10.0680,1.74,249.741,247.874,...,248.071,451.240,468.927,434.610,449.688,484.411,447.826,458.567,66.91,1.31


In [4]:
print(df["date"].dtype)
print(type(df["date"].iloc[0]))

str
<class 'str'>


In [5]:
### 3.1 Análisis de Valores Nulos

In [6]:
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    "Nulos": nulos,
    "Porcentaje %": pct_nulos
})

print(resumen_nulos[resumen_nulos["Nulos"] > 0])
print(f"\nTotal columnas sin nulos: {(nulos == 0).sum()}/{df.shape[1]}")

Empty DataFrame
Columns: [Nulos, Porcentaje %]
Index: []

Total columnas sin nulos: 24/24


### 3.2 Estadísticos Descriptivos

In [7]:
df.describe().round(2)

,% Iron Feed,% Silica Feed,Starch Flow,Amina Flow,Ore Pulp Flow,Ore Pulp pH,Ore Pulp Density,Flotation Column 01 Air Flow,Flotation Column 02 Air Flow,Flotation Column 03 Air Flow,...,Flotation Column 07 Air Flow,Flotation Column 01 Level,Flotation Column 02 Level,Flotation Column 03 Level,Flotation Column 04 Level,Flotation Column 05 Level,Flotation Column 06 Level,Flotation Column 07 Level,% Iron Concentrate,% Silica Concentrate
count,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,...,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00,737453.00
mean,56.29,14.65,2869.14,488.14,397.58,9.77,1.68,280.15,277.16,281.08,...,290.75,520.24,522.65,531.35,420.32,425.25,429.94,421.02,65.05,2.33
std,5.16,6.81,1215.20,91.23,9.70,0.39,0.07,29.62,30.15,28.56,...,28.67,131.01,128.17,150.84,91.79,84.54,89.86,84.89,1.12,1.13
min,42.74,1.31,0.00,241.67,376.25,8.75,1.52,175.51,175.16,176.47,...,185.96,149.22,210.75,126.26,162.20,166.99,155.84,175.35,62.05,0.60
25%,52.67,8.94,2076.32,431.80,394.26,9.53,1.65,250.28,250.46,250.86,...,256.30,416.98,441.88,411.32,356.68,357.65,358.50,356.77,64.37,1.44
50%,56.08,13.85,3018.43,504.39,399.25,9.80,1.70,299.34,296.22,298.70,...,299.01,491.88,495.96,494.32,411.97,408.77,424.66,411.06,65.21,2.00
75%,59.72,19.60,3727.73,553.26,402.97,10.04,1.73,300.15,300.69,300.38,...,301.90,594.11,595.46,601.25,485.55,484.33,492.68,476.46,65.86,3.01
max,65.78,33.40,6300.23,739.54,418.64,10.81,1.85,373.87,375.99,364.35,...,371.59,862.27,828.92,886.82,680.36,675.64,698.86,659.90,68.01,5.53


## 4. Limpieza y Preparación

**Decisiones de limpieza:**
- Conversión de columna `date` de string a datetime
- Verificación de duplicados
- Verificación de rangos físicamente posibles en variables de calidad

No se detectaron valores nulos en el dataset (verificado en sección anterior).

In [8]:
# --- 4.1 Conversión de fecha ---
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d %H:%M:%S")

assert pd.api.types.is_datetime64_any_dtype(df["date"]), \
    "ERROR: La conversión a datetime falló."

print(f"✓ Columna 'date' convertida correctamente")
print(f"  Tipo actual: {df['date'].dtype}")
print(f"  Fecha mínima: {df['date'].min()}")
print(f"  Fecha máxima: {df['date'].max()}")
print(f"  Rango total: {(df['date'].max() - df['date'].min()).days} días")

✓ Columna 'date' convertida correctamente
  Tipo actual: datetime64[us]
  Fecha mínima: 2017-03-10 01:00:00
  Fecha máxima: 2017-09-09 23:00:00
  Rango total: 183 días


In [9]:
# --- 4.2 Duplicados ---
duplicados = df.duplicated().sum()
print(f"\n✓ Filas duplicadas: {duplicados}")

# Las filas con mismo timestamp son normales en datos industriales
# (múltiples lecturas por segundo). No se eliminan sin análisis previo.
duplicados_fecha = df.duplicated(subset=["date"]).sum()
print(f"  Timestamps duplicados: {duplicados_fecha:,}")
print(f"  ({duplicados_fecha/len(df)*100:.1f}% del dataset)")


✓ Filas duplicadas: 1171
  Timestamps duplicados: 733,356
  (99.4% del dataset)


In [10]:
# --- 4.3 Verificación de rangos físicos ---
# Basado en conocimiento del proceso de flotación

checks = {
    "% Iron Concentrate entre 50 y 75": 
        df["% Iron Concentrate"].between(50, 75).all(),
    "% Silica Concentrate entre 0 y 10": 
        df["% Silica Concentrate"].between(0, 10).all(),
    "% Iron Feed entre 40 y 70": 
        df["% Iron Feed"].between(40, 70).all(),
    "Ore Pulp pH entre 7 y 13": 
        df["Ore Pulp pH"].between(7, 13).all()
}

print("\n✓ Verificación de rangos físicos:")
for check, resultado in checks.items():
    estado = "✓ OK" if resultado else "⚠️ FUERA DE RANGO"
    print(f"  {estado} — {check}")


✓ Verificación de rangos físicos:
  ✓ OK — % Iron Concentrate entre 50 y 75
  ✓ OK — % Silica Concentrate entre 0 y 10
  ✓ OK — % Iron Feed entre 40 y 70
  ✓ OK — Ore Pulp pH entre 7 y 13


In [11]:
# --- 4.4 Eliminación de filas completamente duplicadas ---
filas_antes = len(df)
df = df.drop_duplicates()
filas_despues = len(df)

print(f"✓ Filas eliminadas: {filas_antes - filas_despues:,}")
print(f"✓ Filas restantes: {filas_despues:,}")

✓ Filas eliminadas: 1,171
✓ Filas restantes: 736,282


### 4.4 Tratamiento de Duplicados

Se identificaron 1.171 filas completamente duplicadas (las 24 columnas 
idénticas). En un sistema SCADA esto no corresponde a comportamiento 
normal — representa error de exportación. Se eliminan.

Los timestamps duplicados (99.4%) NO se eliminan: corresponden a múltiples 
snapshots del estado de planta en el mismo segundo, comportamiento 
esperado en datos industriales de alta frecuencia.